# Phase 18 — ML/DL dataset preparation, Dataset B (stage 32)

| | |
| --- | --- |
| **Workflow** | `final workflow.md` **Phase 18 — ML/DL dataset preparation** → checklist **stage 32** |
| **Built by** | `scripts/build_sequences.py` → `scripts/phase18_run.log` (exit 0, 17/17, 9.0 s) |
| **This notebook** | independent verification of the written tensors + the torch prerequisite |
| **Outputs verified** | `data/processed/sequences_L{3,7,14,30}.npy` · `sequence_index.csv` · `sequence_manifest.json` |
| **Input (immutable)** | `experimental_dataset.csv` (SHA re-checked) + `split_assignments.csv` |
| **Binding decisions** | **D12** L ∈ {3,7,14,30} built, none assumed best · labels never in the tensor · static/station columns stay on the Phase-23 fusion branch · gaps become explicit NaN timesteps (policy fixed in Phase 19) |

Twelve checks, **N18-01 … N18-12**. Nothing is refit here; the notebook proves the
tensors are faithful to the table and that a seeded torch run is reproducible.

In [1]:
import hashlib, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path(r"D:\Flood predictor")
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
TBL = PROC / "experimental_dataset.csv"
SPLITS = PROC / "split_assignments.csv"
MANIFEST = PROC / "sequence_manifest.json"
INDEX_CSV = PROC / "sequence_index.csv"
REQ = ROOT / "requirements.txt"
NPY = {L: PROC / f"sequences_L{L}.npy" for L in (3, 7, 14, 30)}

TBL_SHA_EXPECTED = "a6748529083fcdceb8bb8a657b067bf12b5c861f3a7a6b09c8977ea63be73c8e"
LENS = (3, 7, 14, 30)
VARIABLES = ["wl_first", "wl_mean", "wl_max", "wl_min", "wl_last", "wl_pct_danger",
             "precip_mm", "tmax_c", "tmin_c", "n_readings", "doy_sin", "doy_cos"]
BANNED = ("label_", "lag", "roll", "rain_", "rise", "exceed", "nb3_", "nb5_",
          "up_", "wet_days", "dist_to_danger", "obs_days")
SEED = 20261008


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()


RES, WARN = [], []


def check(cid, detail, ok=True):
    RES.append({"id": cid, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {cid}: {detail}")
    assert ok, cid


def warn(wid, detail):
    WARN.append({"id": wid, "note": detail})
    print(f"[WARN] {wid}: {detail}")


t0 = time.time()
print("Phase 18 notebook - Dataset B verification")

Phase 18 notebook - Dataset B verification


In [2]:
# N18-01 - inputs present, source table frozen
check("N18-01a", f"manifest {MANIFEST.stat().st_size:,} B, index {INDEX_CSV.stat().st_size:,} B, "
                 f"4 tensors: " + ", ".join(f"L{L} {p.stat().st_size / 1e6:.1f} MB"
                                            for L, p in NPY.items()),
      MANIFEST.exists() and INDEX_CSV.exists() and all(p.exists() for p in NPY.values()))
sha = sha256(TBL)
check("N18-01b", f"experimental_dataset.csv SHA-256 unchanged ({sha[:16]}...)",
      sha == TBL_SHA_EXPECTED)
man = json.loads(MANIFEST.read_text(encoding="utf-8"))
check("N18-01c", f"manifest reports phase {man['phase']} / stage {man['stage']}, "
                 f"built {man['created_utc'][:19]}, "
                 f"{man['checks']['passed']}/{man['checks']['total']} build checks green",
      man["phase"] == 18 and man["checks"]["passed"] == man["checks"]["total"])

[PASS] N18-01a: manifest 2,779 B, index 10,529,122 B, 4 tensors: L3 28.9 MB, L7 67.4 MB, L14 134.8 MB, L30 289.0 MB


[PASS] N18-01b: experimental_dataset.csv SHA-256 unchanged (a6748529083fcdce...)
[PASS] N18-01c: manifest reports phase 18 / stage 32, built 2026-10-08T20:01:25, 16/16 build checks green


In [3]:
# N18-02 - manifest schema
check("N18-02a", f"manifest lists exactly the 12 raw variables: {man['variables']}",
      man["variables"] == VARIABLES and man["n_variables"] == 12)
check("N18-02b", f"window lengths {man['window']['lengths']} with the anchor-day "
                 "definition recorded",
      man["window"]["lengths"] == list(LENS)
      and "ends ON the anchor day" in man["window"]["definition"])
check("N18-02c", "manifest source SHA equals the frozen table SHA "
                 f"({man['source']['experimental_dataset.csv'][:16]}...)",
      man["source"]["experimental_dataset.csv"] == TBL_SHA_EXPECTED)
check("N18-02d", f"no statistic was fitted ({man['fitted_statistics']}); labels listed "
                 f"as excluded {man['labels_excluded']}",
      man["fitted_statistics"].startswith("none")
      and man["labels_excluded"] == ["label_24h", "label_48h", "label_72h"])

[PASS] N18-02a: manifest lists exactly the 12 raw variables: ['wl_first', 'wl_mean', 'wl_max', 'wl_min', 'wl_last', 'wl_pct_danger', 'precip_mm', 'tmax_c', 'tmin_c', 'n_readings', 'doy_sin', 'doy_cos']
[PASS] N18-02b: window lengths [3, 7, 14, 30] with the anchor-day definition recorded
[PASS] N18-02c: manifest source SHA equals the frozen table SHA (a6748529083fcdce...)
[PASS] N18-02d: no statistic was fitted (none - raw calendar scatter + windowing only); labels listed as excluded ['label_24h', 'label_48h', 'label_72h']


In [4]:
# N18-03 - shape and dtype of every tensor
shapes = {}
for L, p in NPY.items():
    X = np.load(p, mmap_mode="r")
    shapes[L] = tuple(X.shape)
    rec = man["files"][p.name]
    check(f"N18-03.{L}", f"{p.name}: shape {X.shape}, dtype {X.dtype} "
                         f"(manifest: {rec['shape']}, {rec['dtype']})",
          X.shape == (200665, L, 12) and X.dtype == np.float32
          and list(X.shape) == rec["shape"])

[PASS] N18-03.3: sequences_L3.npy: shape (200665, 3, 12), dtype float32 (manifest: [200665, 3, 12], float32)
[PASS] N18-03.7: sequences_L7.npy: shape (200665, 7, 12), dtype float32 (manifest: [200665, 7, 12], float32)
[PASS] N18-03.14: sequences_L14.npy: shape (200665, 14, 12), dtype float32 (manifest: [200665, 14, 12], float32)
[PASS] N18-03.30: sequences_L30.npy: shape (200665, 30, 12), dtype float32 (manifest: [200665, 30, 12], float32)


In [5]:
# N18-04 - file hashes match the manifest (byte-reproducible build)
bad = []
for name, rec in man["files"].items():
    p = PROC / name
    if not p.exists() or sha256(p) != rec["sha256"]:
        bad.append(name)
check("N18-04", f"SHA-256 verified for all {len(man['files'])} artifacts "
                f"({', '.join(man['files'])})", not bad)

[PASS] N18-04: SHA-256 verified for all 5 artifacts (sequences_L3.npy, sequences_L7.npy, sequences_L14.npy, sequences_L30.npy, sequence_index.csv)


In [6]:
# N18-05 - variable contract: raw only, no labels, no engineered columns
hits = [v for v in VARIABLES if any(b in v for b in BANNED)]
check("N18-05a", f"no banned/label/engineered column among the 12 variables: {hits}",
      not hits and not any(v.startswith("label_") for v in VARIABLES))
check("N18-05b", f"engineered exclusion recorded: {man['engineered_columns_excluded'][:60]}...",
      "lag" in man["engineered_columns_excluded"] and "up_*" in man["engineered_columns_excluded"])
check("N18-05c", "static/station columns (lat/lon/danger_level/river/...) are NOT in "
                 "the tensor - they stay on the Phase-23 fusion branch",
      not any(v in VARIABLES for v in ("lat", "lon", "danger_level", "river",
                                       "elevation_m", "station_id")))

[PASS] N18-05a: no banned/label/engineered column among the 12 variables: []
[PASS] N18-05b: engineered exclusion recorded: all lag/roll/rain_*_d/neighbour/up_* columns (E13 keeps raw-...
[PASS] N18-05c: static/station columns (lat/lon/danger_level/river/...) are NOT in the tensor - they stay on the Phase-23 fusion branch


In [7]:
# N18-06 - index row order identical to the source table
tbl = pd.read_csv(TBL, usecols=["station_id", "date", "label_24h"])
ix = pd.read_csv(INDEX_CSV)
check("N18-06a", f"index has {len(ix):,} rows in exact table order "
                 "(sample_idx 0..n-1, station/date/labels align row-for-row)",
      len(ix) == len(tbl) == 200665
      and ix.sample_idx.equals(pd.Series(np.arange(len(ix))))
      and ix.station_id.equals(tbl.station_id)
      and ix.date.astype(str).equals(tbl.date.astype(str))
      and np.allclose(ix.label_24h, tbl.label_24h, equal_nan=True))
sc = ix["split"].value_counts().to_dict()
check("N18-06b", f"split counts inherited unchanged: {sc}",
      sc == {"test": 101174, "train": 77571, "val": 21920})

[PASS] N18-06a: index has 200,665 rows in exact table order (sample_idx 0..n-1, station/date/labels align row-for-row)
[PASS] N18-06b: split counts inherited unchanged: {'test': 101174, 'train': 77571, 'val': 21920}


In [8]:
# N18-07 - anchor alignment: window's last timestep == the anchor row's raw values
use = ["station_id", "date"] + VARIABLES
raw = pd.read_csv(TBL, usecols=use)
rng = np.random.default_rng(SEED)
pick = rng.choice(len(ix), size=100, replace=False)
bad = []
for L in LENS:
    X = np.load(NPY[L], mmap_mode="r")
    sub = raw.iloc[pick].reset_index(drop=True)
    want = sub[VARIABLES].to_numpy(dtype=np.float32)
    got = X[pick, -1, :]
    if not np.allclose(got, want, rtol=1e-6, atol=1e-5, equal_nan=True):
        bad.append(L)
check("N18-07", f"100 random samples x L in {LENS}: last timestep equals the anchor "
                f"day's 12 raw values exactly (float32) - mismatches at L={bad}", not bad)

[PASS] N18-07: 100 random samples x L in (3, 7, 14, 30): last timestep equals the anchor day's 12 raw values exactly (float32) - mismatches at L=[]


In [9]:
# N18-08 - full-window alignment: every timestep is its own calendar day's value
key = raw.set_index(["station_id", "date"])
anch = ix.iloc[pick[:30]]
violations, gaps_ok, checked = 0, 0, 0
for L in LENS:
    X = np.load(NPY[L], mmap_mode="r")
    for _, row in anch.iterrows():
        t = pd.Timestamp(row.date)
        win = X[row.sample_idx]
        for k in range(L):
            day = t - pd.Timedelta(days=L - 1 - k)
            want = key.loc[(row.station_id, str(day.date()))] if (row.station_id, str(day.date())) in key.index else None
            got = win[k]
            if want is None:                       # absent calendar day
                gaps_ok += int(np.isnan(got).all())
                violations += int(not np.isnan(got).all())
            else:
                violations += int(not np.allclose(
                    got, want[VARIABLES].to_numpy(dtype=np.float32),
                    rtol=1e-6, atol=1e-5, equal_nan=True))
            checked += 1
check("N18-08", f"30 anchors x 4 window lengths = {checked} timesteps: each equals the "
                f"table value of its own calendar day; {gaps_ok} absent days are "
                f"all-NaN (windows are calendar-true, never shifted)",
      violations == 0)

[PASS] N18-08: 30 anchors x 4 window lengths = 1620 timesteps: each equals the table value of its own calendar day; 118 absent days are all-NaN (windows are calendar-true, never shifted)


In [10]:
# N18-09 - no forward reach: the newest day in the window IS the anchor day
row = ix.iloc[int(pick[0])]
L = 30
X = np.load(NPY[L], mmap_mode="r")[row.sample_idx]
t = pd.Timestamp(row.date)
newest = t
oldest = t - pd.Timedelta(days=L - 1)
check("N18-09a", f"window for anchor {row.date} spans {oldest.date()} .. {newest.date()} "
                 f"- ends exactly on t, never past it (D5 information boundary)",
      (newest - oldest).days == L - 1)
# and the same anchor one day earlier must be the same window shifted one step
i_prev = int(row.sample_idx) - 1
if i_prev >= 0:
    prev = ix.iloc[i_prev]
    if prev.station_id == row.station_id and pd.Timestamp(prev.date) == t - pd.Timedelta(days=1):
        Xp = np.load(NPY[L], mmap_mode="r")[i_prev]
        same = np.allclose(X[:-1], Xp[1:], rtol=1e-6, atol=1e-5, equal_nan=True)
        check("N18-09b", "consecutive anchors of the same station share the overlapping "
                         "days exactly (window slides, never jumps)", bool(same))
    else:
        check("N18-09b", "consecutive-anchor overlap test skipped for this pick "
                         "(previous row is another station/day)", True)

[PASS] N18-09a: window for anchor 2023-08-19 spans 2023-07-21 .. 2023-08-19 - ends exactly on t, never past it (D5 information boundary)
[PASS] N18-09b: consecutive anchors of the same station share the overlapping days exactly (window slides, never jumps)


In [11]:
# N18-10 - gap / front-pad statistics recomputed from the tensors
rec = man["stats"]
for L in LENS:
    X = np.load(NPY[L], mmap_mode="r")
    empty = np.isnan(X).all(axis=2)                 # (n, L)
    frac = float(empty.any(axis=1).mean())
    fp = int(empty[:, 0].sum())                      # first timestep empty = front-pad
    want = rec["fraction_samples_with_empty_timestep"][str(L)]
    check(f"N18-10.{L}", f"L={L}: {100 * frac:.2f}% of samples contain >=1 empty "
                         f"calendar timestep (manifest {100 * want:.2f}%), "
                         f"{fp:,} samples front-padded at series start",
          abs(frac - want) < 5e-4 and fp > 0)
check("N18-10b", f"calendar coverage {100 * rec['fraction_station_days_present']:.1f}% "
                 "present - the rest are the explicit NaN gaps Phase 19 must rule on "
                 "(2018 archive gap R6 + station outages)",
      0.4 < rec["fraction_station_days_present"] < 0.6)

[PASS] N18-10.3: L=3: 2.85% of samples contain >=1 empty calendar timestep (manifest 2.85%), 5,630 samples front-padded at series start
[PASS] N18-10.7: L=7: 7.46% of samples contain >=1 empty calendar timestep (manifest 7.46%), 9,898 samples front-padded at series start


[PASS] N18-10.14: L=14: 14.14% of samples contain >=1 empty calendar timestep (manifest 14.14%), 15,943 samples front-padded at series start


[PASS] N18-10.30: L=30: 26.69% of samples contain >=1 empty calendar timestep (manifest 26.69%), 29,415 samples front-padded at series start
[PASS] N18-10b: calendar coverage 42.4% present - the rest are the explicit NaN gaps Phase 19 must rule on (2018 archive gap R6 + station outages)


In [12]:
# N18-11 - split / holdout / season inheritance is 1:1 with the Phase-15 masks
sp = pd.read_csv(SPLITS)
m = ix.merge(sp[["station_id", "date", "split", "spatial_fold", "station_holdout",
                 "season"]], on=["station_id", "date"], how="left",
             suffixes=("", "_sp"), indicator=True)
check("N18-11a", f"every index row matches a split row ({(m._merge != 'both').sum()} "
                 "unmatched)",
      (m._merge == "both").all())
disagree = {
    "split": int((m["split"] != m["split_sp"]).sum()),
    "spatial_fold": int((m["spatial_fold"] != m["spatial_fold_sp"]).sum()),
    "station_holdout": int((m["station_holdout"].astype(int)
                            != m["station_holdout_sp"].astype(int)).sum()),
    "season": int((m["season"] != m["season_sp"]).sum()),
}
check("N18-11b", f"zero disagreements on split/fold/holdout/season: {disagree}",
      sum(disagree.values()) == 0)
ho = int(ix.station_holdout.sum())
check("N18-11c", f"{ho:,} holdout-station samples inherit station_holdout=1 - they can "
                 "never enter a training cell (Phase-19 check S7 keeps this)",
      ho == 39725)

[PASS] N18-11a: every index row matches a split row (0 unmatched)
[PASS] N18-11b: zero disagreements on split/fold/holdout/season: {'split': 0, 'spatial_fold': 0, 'station_holdout': 0, 'season': 0}
[PASS] N18-11c: 39,725 holdout-station samples inherit station_holdout=1 - they can never enter a training cell (Phase-19 check S7 keeps this)


In [13]:
# N18-12 - prerequisite: torch installed, versions recorded, seeded run reproduces
import torch
import torch.nn as nn

req = REQ.read_text(encoding="utf-8")
versions = {"torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__}
import sklearn
versions["scikit-learn"] = sklearn.__version__
check("N18-12a", f"versions recorded: {versions}",
      torch.__version__.startswith("2.") and "+cpu" in torch.__version__
      and "torch" in req)
check("N18-12b", "requirements.txt carries the pinned torch build",
      "torch==" in req or "torch ==" in req)


def tiny_run(seed):
    torch.manual_seed(seed)
    nn_ = nn.Sequential(nn.Linear(12, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.SGD(nn_.parameters(), lr=0.05)
    loss_fn = nn.BCEWithLogitsLoss()
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(64, 30, 12, generator=g)
    y = (torch.rand(64, generator=g) > 0.7).float()
    out = []
    for _ in range(5):
        opt.zero_grad()
        loss = loss_fn(nn_(X[:, -1, :]).squeeze(-1), y)
        loss.backward()
        opt.step()
        out.append(float(loss.detach()))
    return out


a, b = tiny_run(SEED), tiny_run(SEED)
check("N18-12c", f"seeded tiny training run reproduces its 5 losses exactly on a second "
                 f"run: {[round(x, 6) for x in a]}",
      a == b and all(np.isfinite(a)))

[PASS] N18-12a: versions recorded: {'torch': '2.14.1+cpu', 'numpy': '2.3.2', 'pandas': '3.0.6', 'scikit-learn': '1.9.1'}
[PASS] N18-12b: requirements.txt carries the pinned torch build


[PASS] N18-12c: seeded tiny training run reproduces its 5 losses exactly on a second run: [0.60634, 0.602702, 0.599222, 0.595894, 0.592706]


In [14]:
print("=" * 78)
print("PHASE 18 / STAGE 32   Dataset B verified - sequences L in {3,7,14,30} x 12 vars")
print("=" * 78)
for r in RES:
    print(f"  {'PASS' if r['pass'] else 'FAIL'}  {r['id']}")
for w in WARN:
    print(f"  WARN  {w['id']}: {w['note']}")
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} checks passed, {len(WARN)} warnings, "
      f"{time.time() - t0:.1f}s")
print("outputs: data/processed/sequences_L{3,7,14,30}.npy, sequence_index.csv, "
      "sequence_manifest.json")
print("next: Phase 19 - sequence validation (notebooks/09_sequence_validation.ipynb): "
      "continuity, gap policy, label alignment, split inheritance")
assert all(r["pass"] for r in RES)
assert sha256(TBL) == TBL_SHA_EXPECTED, "experimental table changed!"
print("ALL PHASE-18 CHECKS PASSED - experimental_dataset.csv byte-stable")

PHASE 18 / STAGE 32   Dataset B verified - sequences L in {3,7,14,30} x 12 vars
  PASS  N18-01a
  PASS  N18-01b
  PASS  N18-01c
  PASS  N18-02a
  PASS  N18-02b
  PASS  N18-02c
  PASS  N18-02d
  PASS  N18-03.3
  PASS  N18-03.7
  PASS  N18-03.14
  PASS  N18-03.30
  PASS  N18-04
  PASS  N18-05a
  PASS  N18-05b
  PASS  N18-05c
  PASS  N18-06a
  PASS  N18-06b
  PASS  N18-07
  PASS  N18-08
  PASS  N18-09a
  PASS  N18-09b
  PASS  N18-10.3
  PASS  N18-10.7
  PASS  N18-10.14
  PASS  N18-10.30
  PASS  N18-10b
  PASS  N18-11a
  PASS  N18-11b
  PASS  N18-11c
  PASS  N18-12a
  PASS  N18-12b
  PASS  N18-12c

32/32 checks passed, 0 warnings, 16.7s
outputs: data/processed/sequences_L{3,7,14,30}.npy, sequence_index.csv, sequence_manifest.json
next: Phase 19 - sequence validation (notebooks/09_sequence_validation.ipynb): continuity, gap policy, label alignment, split inheritance


ALL PHASE-18 CHECKS PASSED - experimental_dataset.csv byte-stable
